# Playbook Arsenal: клиенты внутри Assistant

Ноутбук запускает Model Mode `Arsenal` и через каждый клиент по цепочке `llamas → models → assistants → clients` отправляет модели слово «Привет». Каждый пример проверяет, что модель вернула непустой ответ.

In [1]:
# Automatically reload imported modules when their source code changes
%load_ext autoreload
%autoreload 2

# Set the working directory to the ZEMI component root
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

c:\Users\Axoman\Documents\ZEMI\zemilib_tests\tests
c:\Users\Axoman\Documents\ZEMI\zemilib_tests


WindowsPath('c:/Users/Axoman/Documents/ZEMI/zemilib_tests')

In [2]:
import sys
import zemi
from zemi.arsenal import ArsenalSession
from zemi.arsenal.clients import Clients

## Построение Arsenal

Конструктор читает существующую Model Mode-конфигурацию без скачивания и запуска. При создании дерева каждый `Assistant` автоматически получает собственный `Clients`: URL формируется из `llama.host` и `llama.port`, а модель берётся из `model.alias`.

In [3]:
arsenal = ArsenalSession(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml"
)
# arsenal.download()  # Раскомментируйте, чтобы заранее скачать все ресурсы.

assistant = arsenal.llamas.primary.models.qwen.assistants.assistant
clients = assistant.clients

assert isinstance(clients, Clients)
assert clients.server_url == "http://127.0.0.1:8080"
assert clients.openai_url == "http://127.0.0.1:8080/v1"
assert clients.model == "qwen3.5-4b"

{
    "assistant": assistant.name,
    "server_url": clients.server_url,
    "openai_url": clients.openai_url,
    "model": clients.model,
}

{'assistant': 'assistant',
 'server_url': 'http://127.0.0.1:8080',
 'openai_url': 'http://127.0.0.1:8080/v1',
 'model': 'qwen3.5-4b'}

## Собственные clients у каждого ассистента

Ассистенты одной модели обращаются к одному endpoint и alias, но не разделяют изменяемое состояние и кэш ленивых свойств `Clients`. Ассистенты других моделей автоматически получают соответствующий alias.

In [4]:
json_converter = arsenal.llamas.primary.models.qwen.assistants.json_converter

assert assistant.clients is not json_converter.clients
assert assistant.clients.server_url == json_converter.clients.server_url
assert assistant.clients.model == json_converter.clients.model

[
    (item.name, item.clients.server_url, item.clients.model)
    for item in (assistant, json_converter)
]

[('assistant', 'http://127.0.0.1:8080', 'qwen3.5-4b'),
 ('json_converter', 'http://127.0.0.1:8080', 'qwen3.5-4b')]

## Реальные запросы через все клиенты ассистента

Arsenal запускает локальный llama.cpp-сервер. Каждый клиент получает один и тот же запрос «Привет» и ограничивает ответ небольшим числом токенов. Блок `finally` останавливает сервер даже при ошибке одного из клиентов.

In [5]:
from pydantic import BaseModel
from pydantic_ai import Agent
from concurrent.futures import ThreadPoolExecutor
import guidance

BAML_ROOT = PROJECT_ROOT / "tests" / "playbook_arsenal" / "baml_client"
if str(BAML_ROOT) not in sys.path:
    sys.path.insert(0, str(BAML_ROOT))
from baml_client import b

PROMPT = "Привет"
results = {}
failures = {}

class Greeting(BaseModel):
    answer: str

def save(name, value):
    text = str(value).strip()
    assert text, f"{name} вернул пустой ответ"
    results[name] = text
    print(f"PASS {name:16} -> {text[:120]}")

def check(name, operation):
    try:
        save(name, operation())
    except Exception as error:
        reason = f"{type(error).__name__}: {error}"
        failures[name] = reason
        print(f"FAIL {name:16} -> {reason}")

zemi.arsenal.begin(arsenal, stop_before_begin=True, llama_router_mode=False)
arsenal.llamas.primary.models.qwen
try:
    check("openai", lambda: clients.openai.chat.completions.create(model=clients.model, messages=[{"role": "user", "content": PROMPT}], max_tokens=24).choices[0].message.content)
    check("litellm", lambda: clients.litellm.completion(model=clients.model, messages=[{"role": "user", "content": PROMPT}], max_tokens=24).choices[0].message.content)
    check("dspy", lambda: clients.dspy(prompt=PROMPT, max_tokens=24)[0])
    check("instructor", lambda: clients.instructor.chat.completions.create(model=clients.model, response_model=Greeting, messages=[{"role": "user", "content": PROMPT + " Ответь JSON-объектом с полем answer."}], max_tokens=48).answer)
    def run_pydantic_ai():
        with ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(lambda: Agent(clients.pydantic_ai).run_sync(PROMPT).output)
            return future.result()
    check("pydantic_ai", run_pydantic_ai)
    check("baml", lambda: b.Hello(PROMPT, baml_options={"client_registry": clients.baml}))
    check("smolagents", lambda: clients.smolagents.generate([{"role": "user", "content": [{"type": "text", "text": PROMPT}]}], max_tokens=24).content)

    def run_llama_index():
        client = clients.llama_index
        assert client.model == clients.model
        assert client.context_window == 8192
        return client.complete(PROMPT).text
    check("llama_index", run_llama_index)

    def run_httpx():
        response = clients.httpx.post("/v1/chat/completions", json={"model": clients.model, "messages": [{"role": "user", "content": PROMPT}], "max_tokens": 24})
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    check("httpx", run_httpx)

    def run_llama_cpp_agent():
        provider = clients.llama_cpp_agent
        response = clients.httpx.post(provider.server_chat_completion_endpoint, json={"messages": [{"role": "user", "content": PROMPT}], "max_tokens": 24, "stream": False})
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    check("llama_cpp_agent", run_llama_cpp_agent)

    check("outlines", lambda: clients.outlines(PROMPT, max_tokens=24))

    def run_guidance():
        model = clients.guidance
        model.echo = False
        with guidance.user():
            model += PROMPT
        with guidance.assistant():
            model += guidance.gen(name="answer", max_tokens=24)
        return model["answer"]
    check("guidance", run_guidance)
finally:
    zemi.arsenal.end(arsenal, stop_after_end=True)

print("\n" + "=" * 100)
print("ИТОГОВЫЙ ОТЧЁТ ПО КЛИЕНТАМ")
print("=" * 100)
for name in clients.names:
    if name in results:
        print(f"PASS  {name:16}  {results[name][:70]}")
    else:
        print(f"FAIL  {name:16}  {failures.get(name, 'проверка не была выполнена')}")
print("-" * 100)
print(f"Всего: {len(clients.names)} | PASS: {len(results)} | FAIL: {len(failures)}")

{"passed": results, "failed": failures}


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    · не запущен
[2/2] secondary · 127.0.0.1:8081
    · не запущен
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ARSENAL ГОТОВ · MODEL MODE
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
Скачивание и запуск отложены до первого обращения к модели.
Пример: arsenal.llamas["primary"].models["qwen"]
══════════════════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════════════

{'passed': {'openai': 'Привет! Чем могу вам помочь?',
  'litellm': 'Привет! Чем могу вам помочь?',
  'dspy': 'Привет! Чем могу вам помочь?',
  'instructor': 'Привет!',
  'pydantic_ai': 'Привет! Как я могу вам помочь?',
  'baml': 'Привет! Как я могу вам помочь?',
  'smolagents': 'Привет! Как я могу вам помочь сегодня?',
  'llama_index': 'Привет! Чем могу вам помочь?',
  'httpx': 'Привет! Чем могу быть полезен?',
  'llama_cpp_agent': 'Привет! Чем я могу вам помочь?',
  'outlines': 'Привет! Чем могу вам помочь?',
  'guidance': 'Привет! Чем я могу вам помочь?'},
 'failed': {}}